# Bronze — Nested Payments

Build two representations from `globalmart.bronze.order_payments`:

1. **Nested** — one row per order, `payments` array of structs + summary fields
2. **Flattened** — explode nested back to one row per payment line

**Verify:** nested rows = distinct orders; flattened rows = original payment rows.

In [ ]:
import sys

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.ingestion.nested_payments import (
    NestedPaymentsConfig,
    write_nested_payments,
    build_payments_report,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = NestedPaymentsConfig()
print("Source:", config.source_table)

In [ ]:
# Build nested + flattened tables
write_nested_payments(spark, config)
print("Wrote:", config.nested_table)
print("Wrote:", config.flattened_table)

In [ ]:
display(spark.table(config.nested_table).limit(5))

In [ ]:
display(spark.table(config.flattened_table).limit(5))

In [ ]:
# Validation report
import json

report = build_payments_report(spark, config)
print(json.dumps(report, indent=2))

try:
    path = save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/payments_nested_latest.json",
    )
    print(f"\nSaved to: {path}")
except Exception as exc:
    print(f"\nVolume save skipped: {exc}")